# Output Agent — Unit Tests

Tests `build_output_products` and `run()` from `output.ipynb`.

**Run order:**
1. Setup (sys.path + load output notebook)
2. Mock state
3. Individual test cells — run any subset without the model
4. Full inference test — loads Qwen2.5-3B-Instruct (requires GPU/CPU time)

## Setup

In [ ]:
import sys
from pathlib import Path

# Resolve src/ so utils.helper and agents.* are importable
src_dir = str(Path(".").resolve().parent / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Execute output.ipynb in this kernel — makes run() and build_output_products() available
%run ../src/agents/output.ipynb

## Mock state

In [ ]:
MOCK_STATE = {
    "style_profile": (
        "Warm earth tones, relaxed Japanese streetwear silhouettes, "
        "natural textures like linen and washed denim, minimal branding."
    ),
    "ranked_products": [
        {
            "name": "Uniqlo Wide Linen Trousers",
            "score": 0.91,
            "tags": ["linen", "wide-leg", "neutral", "minimalist"],
            "url": "https://uniqlo.com/example",
            "image_url": "https://uniqlo.com/example.jpg",
            "price": "$49.90",
        },
        {
            "name": "Mango Washed Denim Overshirt",
            "score": 0.84,
            "tags": ["denim", "oversized", "earth tone", "texture"],
            "url": "https://mango.com/example",
            "image_url": "https://mango.com/example.jpg",
            "price": "$69.99",
        },
        {
            "name": "COS Relaxed Cotton Jacket",
            "score": 0.78,
            "tags": ["cotton", "boxy", "minimalist", "neutral"],
            "url": "https://cos.com/example",
            "image_url": "",
            "price": "$129.00",
        },
    ],
    "critic_notes": "Filtered 2 products that matched color but had prominent logo branding.",
}

## Tests — no model required

In [ ]:
# test_build_output_products
products = build_output_products(MOCK_STATE["ranked_products"])
assert len(products) == 3
for p in products:
    assert "name" in p
    assert "price" in p
    assert "url" in p
    assert "image_url" in p
    assert "score" in p
    assert "tags" in p
print("PASS  test_build_output_products")

In [ ]:
# test_run_empty_products — early return, no model call
result = run({**MOCK_STATE, "ranked_products": []})
assert "output_text" in result
assert "output_products" not in result  # early return, no products built
print("PASS  test_run_empty_products")

In [ ]:
# test_run_empty_style_profile — early return, no model call
result = run({**MOCK_STATE, "style_profile": ""})
assert "output_text" in result
print("PASS  test_run_empty_style_profile")

## Full inference test — loads Qwen2.5-3B-Instruct

> This cell loads the model and runs a real generation pass. Skip if you only want to validate the helper logic above.

In [ ]:
# test_run_returns_expected_keys
result = run(MOCK_STATE)
assert "output_text" in result, "missing output_text"
assert "output_products" in result, "missing output_products"
assert isinstance(result["output_text"], str)
assert len(result["output_text"]) > 0
assert isinstance(result["output_products"], list)
assert len(result["output_products"]) == 3
print("PASS  test_run_returns_expected_keys")

## Inspect output

In [ ]:
print("--- output_text ---")
print(result["output_text"])

In [ ]:
print("--- output_products ---")
for p in result["output_products"]:
    print(p)